#Sudoku Ranker

This notebook ranks the difficulty of sudoku puzzle based of the human strategies required to solve them. Once the needed strategies have been determined, a feed-forward neural network is used to assign a numerical difficulty rank to the puzzle.

In [1]:
import numpy as np

### Update Candidates After Value Placement

In [3]:
def update_candidates(candidates, row, col, val):
  """
  Updates possible candidates after placing val at (row, col).

  candidates: 9x9x9 ndarray of booleans
  row, col: int (0-8)
  val: int (1-9)
  """
  digit_idx = val - 1

  # Set only placed digit as possible in this cell
  candidates[row, col, :] = False
  candidates[row, col, digit_idx] = True

  # Eliminate val as candidate from same row and column
  candidates[row, :, digit_idx] = False
  candidates[:, col, digit_idx] = False

  # Eliminate val as candidate from subsquare
  box_row_start = (row // 3) * 3
  box_col_start = (col // 3) * 3
  candidates[box_row_start:box_row_start+3, box_col_start:box_col_start+3, digit_idx] = False

  # Restore True for placed value in its own cell
  candidates[row, col, digit_idx] = True


# Human Strategies for Solving Sudoku

In [4]:
def solve_naked_singles(board, candidates):
  """
  Solves the naked singles strategy.

  board: 9x9 ndarray of ints
  candidates: 9x9x9 ndarray of booleans
  """
  can_fill = []

  for i in range(9): # Iterate rows
    for j in range(9): # Iterate columns
      if board[i, j] != 0:
        continue # Cell filled, continue
      num_candidates = 0
      for k in range(9): # Iterate candidates
        if candidates[i, j, k]:
          num_candidates += 1
          digit_idx = k
      if num_candidates == 1: # Only 1 possibility, save
        can_fill.append((i, j, digit_idx))

  # Fill valid cells
  for i, j, k in can_fill:
    board[i, j] = k + 1
    update_candidates(candidates, i, j, k + 1)

  return len(can_fill)


In [ ]:
def solve_hidden_singles